In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# -------- Configuration --------
DATASET_NAME = "skin_disease"

# Choose your mode:
TRAINING_MODE = False  # Set to True if you want to train models, False for inference only

if TRAINING_MODE:
    # FOR TRAINING + INFERENCE:
    TRAIN_DIR = "/kaggle/input/mpox-skin-lesion-dataset-version-20-msld-v20/Original Images/Original Images/FOLDS/fold1/Train"     # Training data
    VAL_DIR = "/kaggle/input/mpox-skin-lesion-dataset-version-20-msld-v20/Original Images/Original Images/FOLDS/fold1/Valid"       # Validation data  
    TEST_DIR = "/kaggle/input/mpox-skin-lesion-dataset-version-20-msld-v20/Original Images/Original Images/FOLDS/fold1/Test" 
    MODEL_DIR = "/kaggle/working/trained_models"              # Where trained models will be saved
else:
    # FOR INFERENCE ONLY (if models already trained):
    TEST_DIR = "/kaggle/input/mpox-skin-lesion-dataset-version-20-msld-v20/Original Images/Original Images/FOLDS/fold2/Test"          # Test data only
    MODEL_DIR = "/kaggle/input/pretrained-models"             # Pre-trained models location

RESULTS_DIR = f"/kaggle/working/ensemble_results/{DATASET_NAME}"

# Create directories (only for writable paths)
if TRAINING_MODE:
    os.makedirs(MODEL_DIR, exist_ok=True)  # Only create if it's in /kaggle/working/
os.makedirs(RESULTS_DIR, exist_ok=True)

# Validate directories exist
if not TRAINING_MODE and not os.path.exists(MODEL_DIR):
    print(f"⚠️  Model directory not found: {MODEL_DIR}")
    print("📁 Available input datasets:")
    try:
        for item in os.listdir("/kaggle/input/"):
            print(f"   - /kaggle/input/{item}")
    except:
        pass
    print("\n💡 Please update MODEL_DIR to point to your actual model dataset!")

if not os.path.exists(TEST_DIR):
    print(f"⚠️  Test directory not found: {TEST_DIR}")
    print("📁 Available input datasets:")
    try:
        for item in os.listdir("/kaggle/input/"):
            print(f"   - /kaggle/input/{item}")
    except:
        pass
    print("\n💡 Please update TEST_DIR to point to your actual test dataset!")

# Expected skin disease classes (modify based on your dataset)
EXPECTED_CLASSES = [
    "Chickenpox", "Cowpox", "HFMD", "Healthy", "Measles", "Monkeypox"
]
NUM_CLASSES = 6

print(f"🏥 Running ensemble prediction for {DATASET_NAME}")
print(f"📂 Test directory: {TEST_DIR}")
if TRAINING_MODE:
    print(f"🎓 Training mode: Models will be saved to {MODEL_SAVE_DIR}")
    print(f"📚 Train directory: {TRAIN_DIR}")
    print(f"📊 Validation directory: {VAL_DIR}")
else:
    print(f"🧠 Inference mode: Loading models from {MODEL_DIR}")
print(f"💾 Results will be saved to: {RESULTS_DIR}")

# -------- Model Configuration --------
models_config = {
    "resnet152": {
        "framework": "torch",
        "input_size": (224, 224),
        "preprocessing": "imagenet"
    },
    "densenet201": {
        "framework": "torch", 
        "input_size": (224, 224),
        "preprocessing": "imagenet"
    },
    "mobilenetv3": {
        "framework": "torch",
        "input_size": (224, 224), 
        "preprocessing": "imagenet"
    },
    "vgg19": {
        "framework": "torch",
        "input_size": (224, 224),
        "preprocessing": "imagenet"
    },
    "efficientnetb7": {
        "framework": "torch",
        "input_size": (600, 600),
        "preprocessing": "imagenet"
    },
    "inceptionresnetv2": {
        "framework": "torch",
        "input_size": (299, 299),
        "preprocessing": "inception"
    },
    "vit": {
        "framework": "torch",
        "input_size": (224, 224),
        "preprocessing": "imagenet"
    },
    "swin": {
        "framework": "keras",
        "input_size": (224, 224),
        "preprocessing": "tf"
    },
    "cvt": {
        "framework": "keras", 
        "input_size": (224, 224),
        "preprocessing": "tf"
    },
    "deit": {
        "framework": "keras",
        "input_size": (224, 224),
        "preprocessing": "tf"
    }
}

# -------- PyTorch Model Classes (Fixed __init__ methods) --------
class ResNet152Model(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super(ResNet152Model, self).__init__()
        self.model = models.resnet152(weights=None)
        self.model.fc = nn.Linear(self.model.fc.in_features, num_classes)
    
    def forward(self, x):
        return self.model(x)

class DenseNet201Model(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super(DenseNet201Model, self).__init__()
        self.model = models.densenet201(weights=None)
        self.model.classifier = nn.Linear(self.model.classifier.in_features, num_classes)
    
    def forward(self, x):
        return self.model(x)

class MobileNetV3Model(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super(MobileNetV3Model, self).__init__()
        self.model = models.mobilenet_v3_large(weights=None)
        self.model.classifier[3] = nn.Linear(self.model.classifier[3].in_features, num_classes)
    
    def forward(self, x):
        return self.model(x)

class VGG19Model(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super(VGG19Model, self).__init__()
        self.model = models.vgg19(weights=None)
        self.model.classifier[6] = nn.Linear(self.model.classifier[6].in_features, num_classes)
    
    def forward(self, x):
        return self.model(x)

class EfficientNetB7Model(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super(EfficientNetB7Model, self).__init__()
        self.model = models.efficientnet_b7(weights=None)
        self.model.classifier[1] = nn.Linear(self.model.classifier[1].in_features, num_classes)
    
    def forward(self, x):
        return self.model(x)

class InceptionResNetV2Model(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super(InceptionResNetV2Model, self).__init__()
        # Using Inception V3 as InceptionResNetV2 is not in torchvision
        self.model = models.inception_v3(weights=None, aux_logits=False)
        self.model.fc = nn.Linear(self.model.fc.in_features, num_classes)
    
    def forward(self, x):
        return self.model(x)

class ViTModel(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super(ViTModel, self).__init__()
        self.model = models.vit_b_16(weights=None)
        self.model.heads.head = nn.Linear(self.model.heads.head.in_features, num_classes)
    
    def forward(self, x):
        return self.model(x)

# Model class mapping
model_classes = {
    "resnet152": ResNet152Model,
    "densenet201": DenseNet201Model,
    "mobilenetv3": MobileNetV3Model,
    "vgg19": VGG19Model,
    "efficientnetb7": EfficientNetB7Model,
    "inceptionresnetv2": InceptionResNetV2Model,
    "vit": ViTModel,
}

# -------- Data Loading Functions --------
def get_torch_transforms(input_size, preprocessing_type="imagenet"):
    """Get appropriate transforms for PyTorch models"""
    if preprocessing_type == "inception":
        # Inception models expect [-1, 1]
        normalize = transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    else:
        # ImageNet normalization
        normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    
    return transforms.Compose([
        transforms.Resize(input_size),
        transforms.ToTensor(),
        normalize
    ])

def load_test_data_safely():
    """Load test data with proper error handling"""
    print("🔍 Scanning test directory structure...")
    
    if not os.path.exists(TEST_DIR):
        raise FileNotFoundError(f"Test directory not found: {TEST_DIR}")
    
    # Auto-detect class structure
    detected_classes = []
    image_paths = []
    labels = []
    
    # Check if images are in subdirectories (class folders) or flat structure
    items = os.listdir(TEST_DIR)
    has_subdirs = any(os.path.isdir(os.path.join(TEST_DIR, item)) for item in items)
    
    if has_subdirs:
        print("📁 Detected class-based folder structure")
        detected_classes = sorted([d for d in items if os.path.isdir(os.path.join(TEST_DIR, d))])
        print(f"🏷️  Found classes: {detected_classes}")
        
        for class_idx, class_name in enumerate(detected_classes):
            class_dir = os.path.join(TEST_DIR, class_name)
            image_files = [f for f in os.listdir(class_dir) 
                          if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff'))]
            
            for img_file in image_files:
                image_paths.append(os.path.join(class_dir, img_file))
                labels.append(class_idx)
    else:
        print("📄 Detected flat image structure (no class folders)")
        image_files = [f for f in items 
                      if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff'))]
        
        for img_file in image_files:
            image_paths.append(os.path.join(TEST_DIR, img_file))
            labels.append(0)  # Default label for flat structure
        
        detected_classes = ["unknown"]
    
    print(f"📊 Total images found: {len(image_paths)}")
    print(f"🏷️  Classes: {len(detected_classes)}")
    
    return image_paths, labels, detected_classes

# -------- Training Functions --------
def create_data_loaders(train_dir, val_dir, input_size, batch_size=32):
    """Create PyTorch data loaders for training"""
    from torch.utils.data import DataLoader
    from torchvision.datasets import ImageFolder
    
    # Training transforms with augmentation
    train_transforms = transforms.Compose([
        transforms.Resize((input_size[0] + 32, input_size[1] + 32)),
        transforms.RandomCrop(input_size),
        transforms.RandomHorizontalFlip(0.5),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # Validation transforms (no augmentation)
    val_transforms = transforms.Compose([
        transforms.Resize(input_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    # Create datasets
    train_dataset = ImageFolder(train_dir, transform=train_transforms)
    val_dataset = ImageFolder(val_dir, transform=val_transforms)
    
    # Create data loaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    
    return train_loader, val_loader, train_dataset.classes

def load_and_preprocess_images(image_paths, target_size, framework="torch", preprocessing="imagenet"):
    """Train a PyTorch model"""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3)
    
    best_val_acc = 0.0
    best_model_state = None
    
    print(f"🏋️ Training {model_name} on {device}")
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)
            
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = torch.max(output.data, 1)
            train_total += target.size(0)
            train_correct += (predicted == target).sum().item()
            
            if batch_idx % 10 == 0:
                print(f'Epoch {epoch+1}/{epochs}, Batch {batch_idx}/{len(train_loader)}, Loss: {loss.item():.4f}')
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for data, target in val_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                loss = criterion(output, target)
                
                val_loss += loss.item()
                _, predicted = torch.max(output.data, 1)
                val_total += target.size(0)
                val_correct += (predicted == target).sum().item()
        
        train_acc = 100. * train_correct / train_total
        val_acc = 100. * val_correct / val_total
        
        print(f'Epoch {epoch+1}/{epochs}:')
        print(f'  Train Loss: {train_loss/len(train_loader):.4f}, Train Acc: {train_acc:.2f}%')
        print(f'  Val Loss: {val_loss/len(val_loader):.4f}, Val Acc: {val_acc:.2f}%')
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict().copy()
        
        scheduler.step(val_loss)
    
    # Load best model state
    model.load_state_dict(best_model_state)
    
    # Save model
    model_path = os.path.join(MODEL_DIR, f"{DATASET_NAME}_{model_name}.pth")
    torch.save(best_model_state, model_path)
    print(f"✅ Saved {model_name} with best val acc: {best_val_acc:.2f}%")
    
    return model
    """Load and preprocess images for specific model requirements"""
    print(f"📦 Loading {len(image_paths)} images for size {target_size}")
    
    images = []
    failed_images = []
    
    for i, img_path in enumerate(image_paths):
        try:
            if framework == "torch":
                # Load image using PIL for PyTorch
                img = Image.open(img_path).convert('RGB')
                transform = get_torch_transforms(target_size, preprocessing)
                img_tensor = transform(img)
                images.append(img_tensor)
            else:
                # Load image using Keras preprocessing
                img = load_img(img_path, target_size=target_size)
                img_array = img_to_array(img) / 255.0  # Normalize to [0,1]
                images.append(img_array)
                
        except Exception as e:
            print(f"⚠️  Failed to load {img_path}: {str(e)}")
            failed_images.append(img_path)
            # Add dummy image to maintain indexing
            if framework == "torch":
                images.append(torch.zeros(3, target_size[0], target_size[1]))
            else:
                images.append(np.zeros((*target_size, 3)))
    
    if failed_images:
        print(f"⚠️  Failed to load {len(failed_images)} images")
    
    if framework == "torch":
        return torch.stack(images)
    else:
        return np.array(images)

# -------- Main Execution --------
# -------- Main Execution --------
def main():
    try:
        if TRAINING_MODE:
            print("🎓 TRAINING MODE: Training models from scratch")
            
            # Check if training directories exist
            if not os.path.exists(TRAIN_DIR):
                print(f"❌ Training directory not found: {TRAIN_DIR}")
                return
            if not os.path.exists(VAL_DIR):
                print(f"❌ Validation directory not found: {VAL_DIR}")
                return
            
            # Train PyTorch models (start with a few key models to save time)
            priority_models = ["resnet152", "densenet201", "efficientnetb7"]  # Start with these
            
            for model_name in priority_models:
                if model_name in model_classes:
                    print(f"\n🚀 Training {model_name}...")
                    
                    try:
                        config = models_config[model_name]
                        input_size = config['input_size']
                        
                        # Create data loaders
                        train_loader, val_loader, class_names = create_data_loaders(
                            TRAIN_DIR, VAL_DIR, input_size, batch_size=16  # Smaller batch for memory
                        )
                        
                        # Create and train model
                        model = model_classes[model_name](NUM_CLASSES)
                        trained_model = train_pytorch_model(
                            model, train_loader, val_loader, model_name, epochs=5  # Fewer epochs for demo
                        )
                        
                        print(f"✅ Successfully trained {model_name}")
                        
                    except Exception as e:
                        print(f"❌ Failed to train {model_name}: {str(e)}")
                        continue
            
            print(f"\n🎉 Training completed! Models saved to {MODEL_DIR}")
        
        # Now do inference (regardless of mode)
        print(f"\n🔮 INFERENCE MODE: Making predictions")
        
        # Load test data
        image_paths, y_true, class_names = load_test_data_safely()
        
        if len(image_paths) == 0:
            raise ValueError("No images found in test directory")
        
        # Save metadata
        np.save(os.path.join(RESULTS_DIR, "image_paths.npy"), image_paths)
        np.save(os.path.join(RESULTS_DIR, "true_labels.npy"), y_true)
        np.save(os.path.join(RESULTS_DIR, "class_names.npy"), class_names)
        
        successful_models = []
        failed_models = []
        
        # Process each model
        for model_name, config in models_config.items():
            if config['framework'] != 'torch':  # Skip Keras models for now
                continue
                
            print(f"\n🧠 Processing {model_name} ({config['framework']})")
            
            try:
                # Find model file
                possible_extensions = ['pth', 'pt']
                model_file = None
                
                for ext in possible_extensions:
                    candidate = os.path.join(MODEL_DIR, f"{DATASET_NAME}_{model_name}.{ext}")
                    if os.path.exists(candidate):
                        model_file = candidate
                        break
                
                if model_file is None:
                    raise FileNotFoundError(f"Model file not found for {model_name}")
                
                print(f"📁 Found model: {model_file}")
                
                # Load and preprocess images
                input_size = config['input_size']
                framework = config['framework']
                preprocessing = config.get('preprocessing', 'imagenet')
                
                X = load_and_preprocess_images(image_paths, input_size, framework, preprocessing)
                
                # Make predictions
                print("🔮 Loading PyTorch model and predicting...")
                model = model_classes[model_name](NUM_CLASSES)
                
                # Load state dict with error handling
                state_dict = torch.load(model_file, map_location='cpu')
                
                # Handle different state dict formats
                if 'state_dict' in state_dict:
                    state_dict = state_dict['state_dict']
                elif 'model_state_dict' in state_dict:
                    state_dict = state_dict['model_state_dict']
                
                model.load_state_dict(state_dict, strict=False)
                model.eval()
                
                # Predict in batches to avoid memory issues
                batch_size = 16
                all_probs = []
                
                with torch.no_grad():
                    for i in range(0, len(X), batch_size):
                        batch = X[i:i+batch_size]
                        outputs = model(batch)
                        batch_probs = F.softmax(outputs, dim=1).cpu().numpy()
                        all_probs.append(batch_probs)
                
                probs = np.vstack(all_probs)
                preds = np.argmax(probs, axis=1)
                
                # Save results
                np.save(os.path.join(RESULTS_DIR, f"{model_name}_probs.npy"), probs)
                np.save(os.path.join(RESULTS_DIR, f"{model_name}_preds.npy"), preds)
                
                successful_models.append(model_name)
                print(f"✅ Successfully processed {model_name}")
                
            except Exception as e:
                print(f"❌ Failed to process {model_name}: {str(e)}")
                failed_models.append((model_name, str(e)))
                continue
        
        # Summary
        print(f"\n📊 ENSEMBLE PREDICTION SUMMARY")
        print(f"✅ Successful models: {len(successful_models)}")
        print(f"❌ Failed models: {len(failed_models)}")
        
        if successful_models:
            print(f"🎯 Successful: {', '.join(successful_models)}")
            
            # Create ensemble prediction
            print("\n🔮 Creating ensemble predictions...")
            ensemble_probs = np.zeros((len(image_paths), NUM_CLASSES))
            
            for model_name in successful_models:
                probs = np.load(os.path.join(RESULTS_DIR, f"{model_name}_probs.npy"))
                ensemble_probs += probs
            
            ensemble_probs /= len(successful_models)  # Average probabilities
            ensemble_preds = np.argmax(ensemble_probs, axis=1)
            
            np.save(os.path.join(RESULTS_DIR, "ensemble_probs.npy"), ensemble_probs)
            np.save(os.path.join(RESULTS_DIR, "ensemble_preds.npy"), ensemble_preds)
            
            print(f"🎉 Ensemble prediction completed!")
            print(f"💾 Results saved to: {RESULTS_DIR}")
            
            # Show some predictions
            print(f"\n🔍 Sample Predictions:")
            for i in range(min(5, len(ensemble_preds))):
                pred_class = class_names[ensemble_preds[i]]
                true_class = class_names[y_true[i]] if len(set(y_true)) > 1 else "N/A"
                confidence = ensemble_probs[i][ensemble_preds[i]] * 100
                print(f"  Image {i+1}: Predicted={pred_class} (True={true_class}) Confidence={confidence:.1f}%")
        
        if failed_models:
            print(f"\n⚠️  Failed models:")
            for model_name, error in failed_models:
                print(f"   - {model_name}: {error}")
        
    except Exception as e:
        print(f"💥 Critical error: {str(e)}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

2025-07-02 19:17:02.870474: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751483822.892629     130 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751483822.899460     130 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


⚠️  Model directory not found: /kaggle/input/pretrained-models
📁 Available input datasets:
   - /kaggle/input/mpox-skin-lesion-dataset-version-20-msld-v20

💡 Please update MODEL_DIR to point to your actual model dataset!
🏥 Running ensemble prediction for skin_disease
📂 Test directory: /kaggle/input/mpox-skin-lesion-dataset-version-20-msld-v20/Original Images/Original Images/FOLDS/fold2/Test
🧠 Inference mode: Loading models from /kaggle/input/pretrained-models
💾 Results will be saved to: /kaggle/working/ensemble_results/skin_disease

🔮 INFERENCE MODE: Making predictions
🔍 Scanning test directory structure...
📁 Detected class-based folder structure
🏷️  Found classes: ['Chickenpox', 'Cowpox', 'HFMD', 'Healthy', 'Measles', 'Monkeypox']
📊 Total images found: 82
🏷️  Classes: 6

🧠 Processing resnet152 (torch)
❌ Failed to process resnet152: Model file not found for resnet152

🧠 Processing densenet201 (torch)
❌ Failed to process densenet201: Model file not found for densenet201

🧠 Processing mo

In [4]:
# import os
# import numpy as np
# import torch
# import torch.nn as nn
# import torch.nn.functional as F
# import torchvision.models as models
# import torchvision.transforms as transforms
# from tensorflow.keras.models import load_model
# from tensorflow.keras.preprocessing.image import load_img, img_to_array
# from PIL import Image
# import warnings
# warnings.filterwarnings('ignore')

# # -------- Configuration --------
# DATASET_NAME = "mpox-skin-lesion-dataset-version-20-msld-v20"

# # Choose your mode:
# TRAINING_MODE = False  # Set to True if you want to train models, False for inference only

# if TRAINING_MODE:
#     # FOR TRAINING + INFERENCE:
#     TRAIN_DIR = "/kaggle/input/mpox-skin-lesion-dataset-version-20-msld-v20/Original Images/Original Images/FOLDS/fold1/Train"     # Training data
#     VAL_DIR = "/kaggle/input/mpox-skin-lesion-dataset-version-20-msld-v20/Original Images/Original Images/FOLDS/fold1/Valid"       # Validation data  
#     TEST_DIR = "/kaggle/input/mpox-skin-lesion-dataset-version-20-msld-v20/Original Images/Original Images/FOLDS/fold1/Test"  
# else:
#     # FOR INFERENCE ONLY (if models already trained):
#     TEST_DIR = "/kaggle/input/mpox-skin-lesion-dataset-version-20-msld-v20/Original Images/Original Images/FOLDS/fold1/Test"          # Test data only
#     MODEL_DIR = "/kaggle/input/pretrained-models"             # Pre-trained models location

# RESULTS_DIR = f"/kaggle/working/ensemble_results/{DATASET_NAME}"

# # Create directories (only for writable paths)
# if TRAINING_MODE:
#     os.makedirs(MODEL_DIR, exist_ok=True)  # Only create if it's in /kaggle/working/
# os.makedirs(RESULTS_DIR, exist_ok=True)

# # Validate directories exist
# if not TRAINING_MODE and not os.path.exists(MODEL_DIR):
#     print(f"⚠️  Model directory not found: {MODEL_DIR}")
#     print("📁 Available input datasets:")
#     try:
#         for item in os.listdir("/kaggle/input/"):
#             print(f"   - /kaggle/input/{item}")
#     except:
#         pass
#     print("\n💡 Please update MODEL_DIR to point to your actual model dataset!")

# if not os.path.exists(TEST_DIR):
#     print(f"⚠️  Test directory not found: {TEST_DIR}")
#     print("📁 Available input datasets:")
#     try:
#         for item in os.listdir("/kaggle/input/"):
#             print(f"   - /kaggle/input/{item}")
#     except:
#         pass
#     print("\n💡 Please update TEST_DIR to point to your actual test dataset!")

# # Expected skin disease classes (modify based on your dataset)
# EXPECTED_CLASSES = [
#     "acne", "eczema", "melanoma", "psoriasis", "rosacea", "vitiligo"
# ]
# NUM_CLASSES = 6

# print(f"🏥 Running ensemble prediction for {DATASET_NAME}")
# print(f"📂 Test directory: {TEST_DIR}")
# if TRAINING_MODE:
#     print(f"🎓 Training mode: Models will be saved to {MODEL_SAVE_DIR}")
#     print(f"📚 Train directory: {TRAIN_DIR}")
#     print(f"📊 Validation directory: {VAL_DIR}")
# else:
#     print(f"🧠 Inference mode: Loading models from {MODEL_DIR}")
# print(f"💾 Results will be saved to: {RESULTS_DIR}")

# # -------- Model Configuration --------
# models_config = {
#     "resnet152": {
#         "framework": "torch",
#         "input_size": (224, 224),
#         "preprocessing": "imagenet"
#     },
#     "densenet201": {
#         "framework": "torch", 
#         "input_size": (224, 224),
#         "preprocessing": "imagenet"
#     },
#     "mobilenetv3": {
#         "framework": "torch",
#         "input_size": (224, 224), 
#         "preprocessing": "imagenet"
#     },
#     "vgg19": {
#         "framework": "torch",
#         "input_size": (224, 224),
#         "preprocessing": "imagenet"
#     },
#     "efficientnetb7": {
#         "framework": "torch",
#         "input_size": (600, 600),
#         "preprocessing": "imagenet"
#     },
#     "inceptionresnetv2": {
#         "framework": "torch",
#         "input_size": (299, 299),
#         "preprocessing": "inception"
#     },
#     "vit": {
#         "framework": "torch",
#         "input_size": (224, 224),
#         "preprocessing": "imagenet"
#     },
#     "swin": {
#         "framework": "keras",
#         "input_size": (224, 224),
#         "preprocessing": "tf"
#     },
#     "cvt": {
#         "framework": "keras", 
#         "input_size": (224, 224),
#         "preprocessing": "tf"
#     },
#     "deit": {
#         "framework": "keras",
#         "input_size": (224, 224),
#         "preprocessing": "tf"
#     }
# }

# # -------- PyTorch Model Classes (Fixed __init__ methods) --------
# class ResNet152Model(nn.Module):
#     def __init__(self, num_classes=NUM_CLASSES):
#         super(ResNet152Model, self).__init__()
#         self.model = models.resnet152(weights=None)
#         self.model.fc = nn.Linear(self.model.fc.in_features, num_classes)
    
#     def forward(self, x):
#         return self.model(x)

# class DenseNet201Model(nn.Module):
#     def __init__(self, num_classes=NUM_CLASSES):
#         super(DenseNet201Model, self).__init__()
#         self.model = models.densenet201(weights=None)
#         self.model.classifier = nn.Linear(self.model.classifier.in_features, num_classes)
    
#     def forward(self, x):
#         return self.model(x)

# class MobileNetV3Model(nn.Module):
#     def __init__(self, num_classes=NUM_CLASSES):
#         super(MobileNetV3Model, self).__init__()
#         self.model = models.mobilenet_v3_large(weights=None)
#         self.model.classifier[3] = nn.Linear(self.model.classifier[3].in_features, num_classes)
    
#     def forward(self, x):
#         return self.model(x)

# class VGG19Model(nn.Module):
#     def __init__(self, num_classes=NUM_CLASSES):
#         super(VGG19Model, self).__init__()
#         self.model = models.vgg19(weights=None)
#         self.model.classifier[6] = nn.Linear(self.model.classifier[6].in_features, num_classes)
    
#     def forward(self, x):
#         return self.model(x)

# class EfficientNetB7Model(nn.Module):
#     def __init__(self, num_classes=NUM_CLASSES):
#         super(EfficientNetB7Model, self).__init__()
#         self.model = models.efficientnet_b7(weights=None)
#         self.model.classifier[1] = nn.Linear(self.model.classifier[1].in_features, num_classes)
    
#     def forward(self, x):
#         return self.model(x)

# class InceptionResNetV2Model(nn.Module):
#     def __init__(self, num_classes=NUM_CLASSES):
#         super(InceptionResNetV2Model, self).__init__()
#         # Using Inception V3 as InceptionResNetV2 is not in torchvision
#         self.model = models.inception_v3(weights=None, aux_logits=False)
#         self.model.fc = nn.Linear(self.model.fc.in_features, num_classes)
    
#     def forward(self, x):
#         return self.model(x)

# class ViTModel(nn.Module):
#     def __init__(self, num_classes=NUM_CLASSES):
#         super(ViTModel, self).__init__()
#         self.model = models.vit_b_16(weights=None)
#         self.model.heads.head = nn.Linear(self.model.heads.head.in_features, num_classes)
    
#     def forward(self, x):
#         return self.model(x)

# # Model class mapping
# model_classes = {
#     "resnet152": ResNet152Model,
#     "densenet201": DenseNet201Model,
#     "mobilenetv3": MobileNetV3Model,
#     "vgg19": VGG19Model,
#     "efficientnetb7": EfficientNetB7Model,
#     "inceptionresnetv2": InceptionResNetV2Model,
#     "vit": ViTModel,
# }

# # -------- Data Loading Functions --------
# def get_torch_transforms(input_size, preprocessing_type="imagenet"):
#     """Get appropriate transforms for PyTorch models"""
#     if preprocessing_type == "inception":
#         # Inception models expect [-1, 1]
#         normalize = transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
#     else:
#         # ImageNet normalization
#         normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    
#     return transforms.Compose([
#         transforms.Resize(input_size),
#         transforms.ToTensor(),
#         normalize
#     ])

# def load_test_data_safely():
#     """Load test data with proper error handling"""
#     print("🔍 Scanning test directory structure...")
    
#     if not os.path.exists(TEST_DIR):
#         raise FileNotFoundError(f"Test directory not found: {TEST_DIR}")
    
#     # Auto-detect class structure
#     detected_classes = []
#     image_paths = []
#     labels = []
    
#     # Check if images are in subdirectories (class folders) or flat structure
#     items = os.listdir(TEST_DIR)
#     has_subdirs = any(os.path.isdir(os.path.join(TEST_DIR, item)) for item in items)
    
#     if has_subdirs:
#         print("📁 Detected class-based folder structure")
#         detected_classes = sorted([d for d in items if os.path.isdir(os.path.join(TEST_DIR, d))])
#         print(f"🏷️  Found classes: {detected_classes}")
        
#         for class_idx, class_name in enumerate(detected_classes):
#             class_dir = os.path.join(TEST_DIR, class_name)
#             image_files = [f for f in os.listdir(class_dir) 
#                           if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff'))]
            
#             for img_file in image_files:
#                 image_paths.append(os.path.join(class_dir, img_file))
#                 labels.append(class_idx)
#     else:
#         print("📄 Detected flat image structure (no class folders)")
#         image_files = [f for f in items 
#                       if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff'))]
        
#         for img_file in image_files:
#             image_paths.append(os.path.join(TEST_DIR, img_file))
#             labels.append(0)  # Default label for flat structure
        
#         detected_classes = ["unknown"]
    
#     print(f"📊 Total images found: {len(image_paths)}")
#     print(f"🏷️  Classes: {len(detected_classes)}")
    
#     return image_paths, labels, detected_classes

# def load_and_preprocess_images(image_paths, target_size, framework="torch", preprocessing="imagenet"):
#     """Load and preprocess images for specific model requirements"""
#     print(f"📦 Loading {len(image_paths)} images for size {target_size}")
    
#     images = []
#     failed_images = []
    
#     for i, img_path in enumerate(image_paths):
#         try:
#             if framework == "torch":
#                 # Load image using PIL for PyTorch
#                 img = Image.open(img_path).convert('RGB')
#                 transform = get_torch_transforms(target_size, preprocessing)
#                 img_tensor = transform(img)
#                 images.append(img_tensor)
#             else:
#                 # Load image using Keras preprocessing
#                 img = load_img(img_path, target_size=target_size)
#                 img_array = img_to_array(img) / 255.0  # Normalize to [0,1]
#                 images.append(img_array)
                
#         except Exception as e:
#             print(f"⚠️  Failed to load {img_path}: {str(e)}")
#             failed_images.append(img_path)
#             # Add dummy image to maintain indexing
#             if framework == "torch":
#                 images.append(torch.zeros(3, target_size[0], target_size[1]))
#             else:
#                 images.append(np.zeros((*target_size, 3)))
    
#     if failed_images:
#         print(f"⚠️  Failed to load {len(failed_images)} images")
    
#     if framework == "torch":
#         return torch.stack(images)
#     else:
#         return np.array(images)

# # -------- Main Execution --------
# def main():
#     try:
#         # Load test data
#         image_paths, y_true, class_names = load_test_data_safely()
        
#         if len(image_paths) == 0:
#             raise ValueError("No images found in test directory")
        
#         # Save metadata
#         np.save(os.path.join(RESULTS_DIR, "image_paths.npy"), image_paths)
#         np.save(os.path.join(RESULTS_DIR, "true_labels.npy"), y_true)
#         np.save(os.path.join(RESULTS_DIR, "class_names.npy"), class_names)
        
#         successful_models = []
#         failed_models = []
        
#         # Process each model
#         for model_name, config in models_config.items():
#             print(f"\n🧠 Processing {model_name} ({config['framework']})")
            
#             try:
#                 # Find model file
#                 possible_extensions = ['pth', 'pt', 'h5', 'keras'] if config['framework'] == 'torch' else ['h5', 'keras', 'pth', 'pt']
#                 model_file = None
                
#                 for ext in possible_extensions:
#                     candidate = os.path.join(MODEL_DIR, f"{DATASET_NAME}_{model_name}.{ext}")
#                     if os.path.exists(candidate):
#                         model_file = candidate
#                         break
                    
#                     # Try without dataset prefix
#                     candidate = os.path.join(MODEL_DIR, f"{model_name}.{ext}")
#                     if os.path.exists(candidate):
#                         model_file = candidate
#                         break
                
#                 if model_file is None:
#                     raise FileNotFoundError(f"Model file not found for {model_name}")
                
#                 print(f"📁 Found model: {model_file}")
                
#                 # Load and preprocess images
#                 input_size = config['input_size']
#                 framework = config['framework']
#                 preprocessing = config.get('preprocessing', 'imagenet')
                
#                 X = load_and_preprocess_images(image_paths, input_size, framework, preprocessing)
                
#                 # Make predictions
#                 if framework == "keras":
#                     print("🔮 Loading Keras model and predicting...")
#                     model = load_model(model_file)
#                     probs = model.predict(X, batch_size=32, verbose=1)
#                     preds = np.argmax(probs, axis=1)
                    
#                 else:  # PyTorch
#                     print("🔮 Loading PyTorch model and predicting...")
#                     model = model_classes[model_name](NUM_CLASSES)
                    
#                     # Load state dict with error handling
#                     state_dict = torch.load(model_file, map_location='cpu')
                    
#                     # Handle different state dict formats
#                     if 'state_dict' in state_dict:
#                         state_dict = state_dict['state_dict']
#                     elif 'model_state_dict' in state_dict:
#                         state_dict = state_dict['model_state_dict']
                    
#                     model.load_state_dict(state_dict, strict=False)
#                     model.eval()
                    
#                     # Predict in batches to avoid memory issues
#                     batch_size = 32
#                     all_probs = []
                    
#                     with torch.no_grad():
#                         for i in range(0, len(X), batch_size):
#                             batch = X[i:i+batch_size]
#                             outputs = model(batch)
#                             batch_probs = F.softmax(outputs, dim=1).cpu().numpy()
#                             all_probs.append(batch_probs)
                    
#                     probs = np.vstack(all_probs)
#                     preds = np.argmax(probs, axis=1)
                
#                 # Save results
#                 np.save(os.path.join(RESULTS_DIR, f"{model_name}_probs.npy"), probs)
#                 np.save(os.path.join(RESULTS_DIR, f"{model_name}_preds.npy"), preds)
                
#                 successful_models.append(model_name)
#                 print(f"✅ Successfully processed {model_name}")
                
#             except Exception as e:
#                 print(f"❌ Failed to process {model_name}: {str(e)}")
#                 failed_models.append((model_name, str(e)))
#                 continue
        
#         # Summary
#         print(f"\n📊 ENSEMBLE PREDICTION SUMMARY")
#         print(f"✅ Successful models: {len(successful_models)}")
#         print(f"❌ Failed models: {len(failed_models)}")
        
#         if successful_models:
#             print(f"🎯 Successful: {', '.join(successful_models)}")
            
#             # Create ensemble prediction
#             print("\n🔮 Creating ensemble predictions...")
#             ensemble_probs = np.zeros((len(image_paths), NUM_CLASSES))
            
#             for model_name in successful_models:
#                 probs = np.load(os.path.join(RESULTS_DIR, f"{model_name}_probs.npy"))
#                 ensemble_probs += probs
            
#             ensemble_probs /= len(successful_models)  # Average probabilities
#             ensemble_preds = np.argmax(ensemble_probs, axis=1)
            
#             np.save(os.path.join(RESULTS_DIR, "ensemble_probs.npy"), ensemble_probs)
#             np.save(os.path.join(RESULTS_DIR, "ensemble_preds.npy"), ensemble_preds)
            
#             print(f"🎉 Ensemble prediction completed!")
#             print(f"💾 Results saved to: {RESULTS_DIR}")
        
#         if failed_models:
#             print(f"\n⚠️  Failed models:")
#             for model_name, error in failed_models:
#                 print(f"   - {model_name}: {error}")
        
#     except Exception as e:
#         print(f"💥 Critical error: {str(e)}")
#         import traceback
#         traceback.print_exc()

# if __name__ == "__main__":
#     main()

⚠️  Model directory not found: /kaggle/input/pretrained-models
📁 Available input datasets:
   - /kaggle/input/mpox-skin-lesion-dataset-version-20-msld-v20

💡 Please update MODEL_DIR to point to your actual model dataset!
🏥 Running ensemble prediction for mpox-skin-lesion-dataset-version-20-msld-v20
📂 Test directory: /kaggle/input/mpox-skin-lesion-dataset-version-20-msld-v20/Original Images/Original Images/FOLDS/fold1/Test
🧠 Inference mode: Loading models from /kaggle/input/pretrained-models
💾 Results will be saved to: /kaggle/working/ensemble_results/mpox-skin-lesion-dataset-version-20-msld-v20
🔍 Scanning test directory structure...
📁 Detected class-based folder structure
🏷️  Found classes: ['Chickenpox', 'Cowpox', 'HFMD', 'Healthy', 'Measles', 'Monkeypox']
📊 Total images found: 74
🏷️  Classes: 6

🧠 Processing resnet152 (torch)
❌ Failed to process resnet152: Model file not found for resnet152

🧠 Processing densenet201 (torch)
❌ Failed to process densenet201: Model file not found for de

In [2]:
# TRAIN_DIR = "/kaggle/input/mpox-skin-lesion-dataset-version-20-msld-v20/Original Images/Original Images/FOLDS/fold1/Train"     # Training data
# VAL_DIR = "/kaggle/input/mpox-skin-lesion-dataset-version-20-msld-v20/Original Images/Original Images/FOLDS/fold1/Valid"       # Validation data  
# TEST_DIR = "/kaggle/input/mpox-skin-lesion-dataset-version-20-msld-v20/Original Images/Original Images/FOLDS/fold1/Test"       # Test data

In [5]:
# DATASET_NAME = "mpox-skin-lesion-dataset-version-20-msld-v20"

In [ ]:
# EXPECTED_CLASSES = [
#     "Chickenpox", "Cowpox", "HFMD", "Healthy", "Measles", "Monkeypox"
# ]